In [1]:
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from metrics import map_at_k
from text_processing import clean_html, tokenize

In [2]:
arts = pd.read_feather('../candidate_public/candidate_data/articles.f')
cal = pd.read_feather('../candidate_public/candidate_data/calibration.f')
gt = {r.query_id:{int(x) for x in r.ground_truth.split()} for r in cal.itertuples()}
ids = arts.article_id.tolist()
doc = (arts.title + ' ') * 3 + arts.body.map(clean_html)   # заголовок усилен

/home/d_tsyp/projects/support-article-ranking/.venv/lib/python3.14/site-packages/pandas/io/feather_format.py:178: FutureWarning: pyarrow.feather.read_table is deprecated as of 24.0.0. Use pyarrow.ipc.open_file() / RecordBatchFileReader instead. Feather V2 is the Arrow IPC file format.
  pa_table = feather.read_table(


In [3]:
# строим матрицу слова×документы (TF-IDF)
tok = lambda s: tokenize(s, stem=True)
vec = TfidfVectorizer(analyzer=tok, sublinear_tf=True, min_df=2)
A = vec.fit_transform(doc)         # 793 x словарь
print(f'матрица слова×документы: {A.shape}')

матрица слова×документы: (793, 5285)


In [4]:
# SVD 100 тем
svd = TruncatedSVD(n_components=100, random_state=42)
A_topics = normalize(svd.fit_transform(A))   # 793 x 100
print(f'после LSA (документы в темах): {A_topics.shape}')
print(f'сохранено дисперсии 100 темами: {svd.explained_variance_ratio_.sum()*100:.0f}%')

после LSA (документы в темах): (793, 100)
сохранено дисперсии 100 темами: 48%


In [5]:
# запросы в то же пространство, косинус
Q = normalize(svd.transform(vec.transform(cal.query_text)))
S = cosine_similarity(Q, A_topics)
preds = {r.query_id:[ids[i] for i in np.argsort(S[qi])[::-1][:10]] for qi,r in enumerate(cal.itertuples())}
print()
print(f'LSA сам по себе:  MAP@10 = {map_at_k(preds, gt):.4f}   (BM25 был 0.2985)')


LSA сам по себе:  MAP@10 = 0.2842   (BM25 был 0.2985)


In [7]:
from rank_bm25 import BM25Okapi

id2t = dict(zip(arts.article_id, arts.title))

# BM25
tok0 = lambda s: tokenize(s, stem=False)
bm = BM25Okapi([tok0(t) for t in doc])

def rank_of(scores, target):
    order = np.argsort(scores)[::-1]
    pos = {ids[order[p]]:p+1 for p in range(len(order))}
    return min(pos[a] for a in target)

# ищем примеры: BM25 промахнулся (>10), LSA поднял в топ-10
examples = ['Здравствуйте! Меня в пункте выдачи ждет заказ. Если я его не забираю, то мне деньги вернуться на карту?',
            'Как отправить безопастно ювелирное изделие?',
            'здравствуйте !сделал заказ,вернул его,но деньги не пришли']
for q in examples:
    row = cal[cal.query_text==q]
    if row.empty: continue
    qid = row.iloc[0].query_id
    sb = bm.get_scores(tok0(q))
    sl = cosine_similarity(normalize(svd.transform(vec.transform([q]))), A_topics)[0]
    rb, rl = rank_of(sb, gt[qid]), rank_of(sl, gt[qid])
    print('Запрос:', q[:90])
    print('правильная:', ' | '.join(id2t[a] for a in gt[qid]))
    print(f'позиция у BM25: {rb:>3} позиция у LSA: {rl:>3}')

Запрос: Здравствуйте! Меня в пункте выдачи ждет заказ. Если я его не забираю, то мне деньги вернут
правильная: Всё про отмену заказа
позиция у BM25:  15    позиция у LSA:   5
Запрос: Как отправить безопастно ювелирное изделие?
правильная: Упаковать товар
позиция у BM25:  72    позиция у LSA:  11
Запрос: здравствуйте !сделал заказ,вернул его,но деньги не пришли
правильная: Баланс для покупок | Покупателю
позиция у BM25:  14    позиция у LSA:   6


Видно, что LSA ловит то, что BM25 пропускает